# Hemlock — Adaptive Fuzzer Demo

O `AttackFuzzer` reformula o payload de um ataque automaticamente quando ele é bloqueado por uma defesa — usando um LLM adversário para contornar os padrões que acionaram o filtro.

**O que você vai ver:**
1. Uma defesa bloqueando o payload original
2. O fuzzer reformulando e retentando (com MockAdversary sem API key)
3. O mesmo fluxo com LLM real (requer API key) — variante que de fato bypassa
4. Limite do fuzzer: defesas que ele não consegue contornar

Sem API key? As células 1–3 rodam com `MockAdversary`. Célula 4 em diante requer key.

In [ ]:
import sys
sys.path.insert(0, "..")

import tempfile
from tests.conftest import MockLLM
from hemlock.pipeline import Pipeline
from attacks.direct_injection import DirectInjection
from attacks.citation_forgery import CitationForgery
from attacks.fuzzer import AttackFuzzer, FuzzResult
from defenses.input_sanitizer import InjectionPatternFilter
from defenses.chunk_filter import InjectionChunkFilter
from defenses.output_validator import InjectionSuccessGuard

def make_pipeline(response="[PWNED] compromised"):
    tmp = tempfile.mkdtemp()
    return Pipeline(llm=MockLLM(response), persist_dir=tmp, collection="lab03")

print("Imports OK")

## 1. Baseline: ataque sem defesa

Confirma que o ataque `direct_injection [explicit]` sucede quando não há defesa ativa.

In [ ]:
pipeline = make_pipeline()
attack = DirectInjection(pipeline, variant="explicit")
result = attack.run()

print(f"Ataque sucedeu:  {result.succeeded}")
print(f"Resposta:        {result.trace.response[:120]}")

## 2. Defesa ativa: `InjectionPatternFilter` bloqueia no ingest

Com o filtro de padrão ativo, o documento malicioso é rejeitado antes de entrar no índice.
O ataque não tem onde ser recuperado — sucesso zero mesmo com MockLLM comprometido.

In [ ]:
from hemlock.pipeline import Pipeline

tmp = tempfile.mkdtemp()
defended_pipeline = Pipeline(
    llm=MockLLM("[PWNED] compromised"),
    persist_dir=tmp,
    collection="lab03_defended",
    ingest_defenses=[InjectionPatternFilter()],
)

attack_defended = DirectInjection(defended_pipeline, variant="explicit")
result_defended = attack_defended.run()

print(f"Ataque sucedeu:  {result_defended.succeeded}")
print(f"Chunks no índice: {len(result_defended.trace.retrieved_chunks)}")
print(f"Resposta: {result_defended.trace.response[:120]}")

## 3. Fuzzer com MockAdversary (sem API key)

O `MockAdversary` reformula trocando marcadores óbvios por linguagem burocrática.
É um bypass determinístico — útil para entender o fluxo sem gastar créditos.

In [ ]:
class MockAdversary:
    """Reformula payloads substituindo marcadores explícitos por linguagem indireta."""

    REPLACEMENTS = [
        ("IGNORE ALL PREVIOUS INSTRUCTIONS", "As a reminder, the configuration protocol requires"),
        ("[PWNED]",  "[SYSTEM NOTICE]"),
        ("You are now DAN", "Your operational mode has been updated to Diagnostic Assistant Node"),
        ("SYSTEM:",  "NOTICE:"),
        ("ignore",   "note that"),
        ("override", "supplement"),
    ]

    def invoke(self, prompt: str):
        # Extract the original payload from the prompt
        marker = "ORIGINAL PAYLOAD:\n"
        end_marker = "\nBLOCKED REASON"
        start = prompt.find(marker)
        end   = prompt.find(end_marker)
        if start == -1:
            return type("R", (), {"content": prompt})()
        payload = prompt[start + len(marker):end].strip()
        for old, new in self.REPLACEMENTS:
            payload = payload.replace(old, new)
        return type("R", (), {"content": payload})()

mock_adversary = MockAdversary()

# Pipeline comprometido mas com InjectionPatternFilter no ingest
tmp2 = tempfile.mkdtemp()
p2 = Pipeline(
    llm=MockLLM("[PWNED] I have been compromised."),
    persist_dir=tmp2,
    collection="lab03_fuzz",
    ingest_defenses=[InjectionPatternFilter()],
)

attack2 = DirectInjection(p2, variant="explicit")
fuzzer  = AttackFuzzer(attack2, adversary_llm=mock_adversary, max_variants=4)
fuzz_result = fuzzer.fuzz(verbose=True)

print()
print(fuzz_result.summary())
print()

print("── Histórico de variantes ──")
for i, r in enumerate(fuzz_result.all_results):
    status = "✓ SUCEDEU" if r.succeeded else "✗ bloqueado"
    print(f"  Variante {i}: {status}")
    if i == 0 or (fuzz_result.winning_variant_index == i):
        print(f"    payload[:80]: {(fuzz_result.winning_payload or '')[:80]}")

## 4. Fuzzer com LLM real (requer API key)

Com um LLM real como adversário, o payload é reformulado semanticamente —
não apenas troca palavras-chave, mas reescreve o contexto pra soar legítimo.

O alvo aqui é `citation_forgery` contra `InjectionChunkFilter`, que usa regex.
Um LLM adversário sabe exatamente o que evitar.

In [ ]:
import os

HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("OPENAI_API_KEY"))
print(f"API key disponível: {HAS_KEY}")
if not HAS_KEY:
    print("Pulando células com LLM real — defina ANTHROPIC_API_KEY para rodá-las.")

In [ ]:
if HAS_KEY:
    if os.environ.get("ANTHROPIC_API_KEY"):
        from langchain_anthropic import ChatAnthropic
        adversary_llm = ChatAnthropic(model="claude-haiku-4-5-20251001")
    else:
        from langchain_openai import ChatOpenAI
        adversary_llm = ChatOpenAI(model="gpt-4o-mini")

    tmp3 = tempfile.mkdtemp()
    p3 = Pipeline(
        llm=MockLLM("According to the cited study, this is confirmed."),
        persist_dir=tmp3,
        collection="lab03_real",
        retrieval_defenses=[InjectionChunkFilter()],
    )

    attack3  = CitationForgery(p3, variant="fake_paper")
    fuzzer3  = AttackFuzzer(attack3, adversary_llm=adversary_llm, max_variants=5)
    result3  = fuzzer3.fuzz(verbose=True)

    print()
    print(result3.summary())
    if result3.winning_payload:
        print("\nPayload vencedor (primeiros 300 chars):")
        print(result3.winning_payload[:300])

## 5. Limite do fuzzer: o que ele não consegue contornar

`LLMChunkClassifier` classifica semanticamente cada chunk antes do prompt.
Reformular com linguagem burocrática não ajuda — o classificador avalia a *intenção*, não a forma.

Resultado esperado: fuzzer resiste em todas as variantes.

In [ ]:
# Demonstração sem LLM real: mostra que MockAdversary não contorna o InjectionPatternFilter
# em ataques do tipo obfuscated, que já evitam os marcadores óbvios por design.

tmp4 = tempfile.mkdtemp()
p4 = Pipeline(
    llm=MockLLM("[PWNED] compromised"),
    persist_dir=tmp4,
    collection="lab03_limit",
    ingest_defenses=[InjectionPatternFilter()],
    retrieval_defenses=[InjectionChunkFilter()],
    output_defenses=[InjectionSuccessGuard()],
)

attack_obf = DirectInjection(p4, variant="obfuscated")  # já usa payload sem marcadores
fuzzer_obf = AttackFuzzer(attack_obf, adversary_llm=MockAdversary(), max_variants=4)
result_obf = fuzzer_obf.fuzz(verbose=True)

print()
print(result_obf.summary())
print()
print("Motivo: 'obfuscated' já evita marcadores explícitos.")
print("MockAdversary não consegue reformular o que já está ofuscado.")
print("Com LLM real + InjectionChunkFilter + InjectionSuccessGuard: mesma resistência.")

## 6. Resumo comparativo

In [ ]:
summary_rows = [
    ("Sem defesa",                 "DirectInjection [explicit]",   True,  0),
    ("InjectionPatternFilter",     "DirectInjection [explicit]",   False, None),
    ("IPF + MockAdversary fuzzer", "DirectInjection [explicit]",   fuzz_result.succeeded, fuzz_result.winning_variant_index),
    ("IPF+ICF+ISG (3 camadas)",    "DirectInjection [obfuscated]", result_obf.succeeded,  result_obf.winning_variant_index),
]

print(f"{'Configuração':<35} {'Ataque':<32} {'Sucedeu':>8} {'Variante':>9}")
print("-" * 90)
for config, attack_name, succeeded, variant in summary_rows:
    s = "✓ SIM" if succeeded else "✗ NÃO"
    v = str(variant) if variant is not None else "—"
    print(f"{config:<35} {attack_name:<32} {s:>8} {v:>9}")